### SPARK STREAMING

In [0]:
import datetime

from utils.logger import logger
from datetime import datetime
from pyspark.sql.functions import current_timestamp

In [0]:
entities = ["customers", "drivers", "locations", "payments", "trips",  "vehicles"]

In [0]:
# Process each entity from the source data
for entity in entities:

    logger.info("Starting Bronze ingestion for entity: %s", entity)

    # Read a sample batch to infer the schema
    logger.info(
        "Inferring schema from source files for entity: %s",
        entity
    )

    df_batch = spark.read.format("csv") \
         .option("header", "True") \
         .option("InferSchema", "True") \
         .load(f"/Volumes/ride_analytics/source/source_data/{entity}/")

    # Store the inferred schema for streaming ingestion
    schema_entity = df_batch.schema

    logger.info(
        "Schema inference completed for entity: %s",
        entity
    )

    # Read new CSV files using Spark Structured Streaming
    logger.info(
        "Starting streaming ingestion for entity: %s",
        entity
    )

    df = spark.readStream.format("csv") \
        .option("header", True) \
        .schema(schema_entity) \
        .load(f"/Volumes/ride_analytics/source/source_data/{entity}/")

    

    # Write incoming records to the Bronze Delta table
    # Trigger once to process the currently available files
    # Checkpoint the stream to track processed files

    # Add ingestion timestamp
    df = df.withColumn(
        "ingestion_timestamp",
        current_timestamp()
    )
    df.writeStream.format("delta") \
        .outputMode("append") \
        .option(
            "checkpointLocation",
            f"/Volumes/ride_analytics/bronze/checkpoint/{entity}"
        ) \
        .trigger(once=True) \
        .toTable(f"ride_analytics.bronze.{entity}")

    logger.info(
        "Bronze ingestion completed successfully for entity: %s",
        entity
    )

2026-09-10 19:28:54,734 - INFO - Starting Bronze ingestion for entity: customers
2026-09-10 19:28:54,738 - INFO - Inferring schema from source files for entity: customers
2026-09-10 19:28:56,173 - INFO - Schema inference completed for entity: customers
2026-09-10 19:28:56,173 - INFO - Starting streaming ingestion for entity: customers
2026-09-10 19:28:59,062 - INFO - Bronze ingestion completed successfully for entity: customers
2026-09-10 19:28:59,063 - INFO - Starting Bronze ingestion for entity: drivers
2026-09-10 19:28:59,063 - INFO - Inferring schema from source files for entity: drivers
2026-09-10 19:29:00,090 - INFO - Schema inference completed for entity: drivers
2026-09-10 19:29:00,091 - INFO - Starting streaming ingestion for entity: drivers
2026-09-10 19:29:03,027 - INFO - Bronze ingestion completed successfully for entity: drivers
2026-09-10 19:29:03,027 - INFO - Starting Bronze ingestion for entity: locations
2026-09-10 19:29:03,028 - INFO - Inferring schema from source fil